# Physics-Informed Neural Network (PINN) for Flettner Rotor Aerodynamics

This notebook contains the complete Physics-Informed Neural Network (PINN) surrogate model framework for Flettner rotating cylinders. It covers dataset creation, physics-informed data augmentation, deep neural network architecture design, regularized loss optimization, validation plotting, streamline flow grid reconstructions, and vorticity contour maps.

Developed as part of the Bachelor of Technology Thesis in Mechanical Engineering at **National Institute of Technology Durgapur**:
* **Shubham Kumar** (Roll: 22ME8107)
* **Rahul Yadav** (Roll: 22ME8064)

Under the supervision of **Dr. Saif Akram** (Department of Mechanical Engineering, NIT Durgapur).

In [ ]:
# ==========================================
#         STEP 1: IMPORT LIBRARIES
# ==========================================
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print("TensorFlow version:", tf.__version__)
print("Eager execution:", tf.executing_eagerly())

In [ ]:
# ==========================================
#   STEP 2: ROTATIONAL FLOW FIELD FORMULAS
# ==========================================

def vortex_velocity(X, Y, x0, y0, gamma, R, core_radius=0.2):
    """
    Computes velocity components (u, v) induced by a point vortex at (x0, y0) with circulation gamma,
    including its image vortex inside the cylinder to maintain the wall boundary condition.
    """
    r_sq = (X - x0)**2 + (Y - y0)**2
    denom1 = r_sq + core_radius**2
    u1 = -gamma / (2 * np.pi) * (Y - y0) / denom1
    v1 = gamma / (2 * np.pi) * (X - x0) / denom1
    
    r0_sq = x0**2 + y0**2
    xi = R**2 * x0 / r0_sq
    yi = R**2 * y0 / r0_sq
    ri_sq = (X - xi)**2 + (Y - yi)**2
    denom2 = ri_sq + core_radius**2
    u2 = -(-gamma) / (2 * np.pi) * (Y - yi) / denom2
    v2 = (-gamma) / (2 * np.pi) * (X - xi) / denom2
    
    return u1 + u2, v1 + v2

def get_flow_velocity(X, Y, alpha, re_val, cl_pinn, cd_pinn, R=1.0, U_inf=1.0):
    """
    Computes velocity field (u, v) around a rotating cylinder combining potential flow,
    clockwise boundary circulation (slip-scaled), downstream wake coordinate bending,
    and point vortices in the wake for separation bubble reconstruction.
    """
    log_re = np.log10(max(re_val, 100))
    
    # 1. Bending coordinate transformation to curve wake downstream
    x_offset = np.clip(X - R, 0, None)
    deflection_max = 0.28 * alpha * (1.0 + 0.5 / log_re)
    deflection_curve = deflection_max * (1.0 - np.exp(-0.45 * x_offset))
    
    Y_bent = Y - deflection_curve
    
    # 2. Potential flow using physical circulation scaled for slip
    slip = 0.75
    gamma_cyl = -2.0 * np.pi * R * U_inf * alpha * slip
    
    r_sq = X**2 + Y_bent**2
    u_pot = U_inf * (1.0 - R**2 * (X**2 - Y_bent**2) / (r_sq**2 + 1e-5)) - (gamma_cyl / (2.0 * np.pi)) * Y_bent / (r_sq + 1e-5)
    v_pot = -U_inf * R**2 * (2.0 * X * Y_bent) / (r_sq**2 + 1e-5) + (gamma_cyl / (2.0 * np.pi)) * X / (r_sq + 1e-5)
    
    # 3. Wake vortices to form recirculation bubbles
    wake_length = 0.25 + 0.8 / log_re
    wake_width = 0.28 - 0.04 * (log_re - 2.0) / 4.0
    wake_width = max(0.12, min(0.35, wake_width))
    
    suppression = np.exp(-(alpha / 4.0)**2)
    gamma_vortex = 4.0 * U_inf * R * cd_pinn * suppression
    
    x1_val = R + wake_length * R
    y1_val = wake_width * R
    x2_val = R + wake_length * R
    y2_val = -wake_width * R
    
    u_w1, v_w1 = vortex_velocity(X, Y_bent, x1_val, y1_val, -gamma_vortex, R, core_radius=0.10)
    u_w2, v_w2 = vortex_velocity(X, Y_bent, x2_val, y2_val, gamma_vortex, R, core_radius=0.10)
    
    u = u_pot + u_w1 + u_w2
    v = v_pot + v_w1 + v_w2
    
    # Rotate velocity vector to align with the bending curve tangent
    dy_dx = np.zeros_like(X)
    mask = X > R
    dy_dx[mask] = deflection_max * 0.45 * np.exp(-0.45 * x_offset[mask])
    theta = np.arctan(dy_dx)
    
    u_rot = u * np.cos(theta) - v * np.sin(theta)
    v_rot = u * np.sin(theta) + v * np.cos(theta)
    
    # Mask inside cylinder
    inside = r_sq < R**2
    u_rot[inside] = np.nan
    v_rot[inside] = np.nan
    
    # Compute vortex center positions in normal coordinates for plotting
    x_vort_offset = np.clip(x1_val - R, 0, None)
    deflection_vort = deflection_max * (1.0 - np.exp(-0.45 * x_vort_offset))
    
    v1 = (x1_val, y1_val + deflection_vort)
    v2 = (x2_val, y2_val + deflection_vort)
    
    return u_rot, v_rot, v1, v2

In [ ]:
# ==========================================
#       STEP 3: RAW LITERATURE DATASETS
# ==========================================

raw_literature_data = {
  "60000": {
    "source": "Aoki & Ito (2001)",
    "alpha": [0.0, 0.5, 1.0, 1.5, 2.0],
    "Cd": [1.200, 1.100, 0.900, 0.600, 0.250],
    "Cl": [0.000, 0.600, 1.500, 2.700, 3.500]
  },
  "140000": {
    "source": "Karabelas (2010)",
    "alpha": [0.0, 0.5, 1.0, 1.3, 1.5, 2.0],
    "Cd": [1.030, 0.950, 0.750, 0.550, 0.350, 0.130],
    "Cl": [0.000, 0.550, 1.300, 2.000, 2.600, 3.400]
  },
  "500000": {
    "source": "Estimated Trend",
    "alpha": [0.0, 0.5, 1.0, 1.5, 2.0],
    "Cd": [0.800, 0.750, 0.600, 0.400, 0.120],
    "Cl": [0.000, 0.500, 1.200, 1.900, 3.200]
  },
  "1000000": {
    "source": "Karabelas et al. (2012)",
    "alpha": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0],
    "Cd": [0.50, 0.45, 0.38, 0.30, 0.28, 0.27, 0.27, 0.28, 0.29],
    "Cl": [0.00, 1.10, 2.20, 3.50, 4.50, 5.20, 5.70, 5.85, 5.85]
  },
  "5000000": {
    "source": "Karabelas et al. (2012)",
    "alpha": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0],
    "Cd": [0.30, 0.28, 0.24, 0.20, 0.18, 0.17, 0.17, 0.18, 0.19],
    "Cl": [0.00, 1.00, 2.10, 3.40, 4.40, 5.10, 5.65, 5.80, 5.85]
  }
}

In [ ]:
# ==========================================
#     STEP 4: DATA AUGMENTATION PIPELINE
# ==========================================

def build_re_block(alpha_pts, cd_pts, cl_pts, re_value, n_interp=60, noise_copies=5, noise_std_cd=0.015, noise_std_cl=0.030):
    alpha_pts = np.array(alpha_pts)
    cd_pts = np.array(cd_pts)
    cl_pts = np.array(cl_pts)
    
    alpha_dense = np.linspace(alpha_pts.min(), alpha_pts.max(), n_interp)
    cd_dense = np.interp(alpha_dense, alpha_pts, cd_pts)
    cl_dense = np.interp(alpha_dense, alpha_pts, cl_pts)
    
    rows = []
    for i in range(len(alpha_dense)):
        rows.append([alpha_dense[i], re_value, cd_dense[i], cl_dense[i]])
        
        # Mirroring symmetry mapping
        if abs(alpha_dense[i]) > 0.05:
            rows.append([-alpha_dense[i], re_value, cd_dense[i], -cl_dense[i]])
            
        # Gaussian noise multiplication
        for _ in range(noise_copies):
            cd_noisy = max(0.05, cd_dense[i] + np.random.normal(0, noise_std_cd))
            cl_noisy = cl_dense[i] + np.random.normal(0, noise_std_cl)
            rows.append([alpha_dense[i], re_value, cd_noisy, cl_noisy])
            
            if abs(alpha_dense[i]) > 0.05:
                cd_noisy_sym = max(0.05, cd_dense[i] + np.random.normal(0, noise_std_cd))
                cl_noisy_sym = -cl_dense[i] + np.random.normal(0, noise_std_cl)
                rows.append([-alpha_dense[i], re_value, cd_noisy_sym, cl_noisy_sym])
                
    return pd.DataFrame(rows, columns=['alpha', 'Re', 'Cd', 'Cl'])

def prepare_dataset(raw_data, n_interp=60, noise_copies=5, split_ratio=0.8, seed=42):
    np.random.seed(seed)
    all_blocks = []
    for re_str, data in raw_data.items():
        re_val = int(re_str)
        df_block = build_re_block(
            alpha_pts=data['alpha'],
            cd_pts=data['Cd'],
            cl_pts=data['Cl'],
            re_value=re_val,
            n_interp=n_interp,
            noise_copies=noise_copies
        )
        all_blocks.append(df_block)
        
    df = pd.concat(all_blocks, ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    X = df[['alpha', 'Re']].values.astype(np.float32)
    Y = df[['Cd', 'Cl']].values.astype(np.float32)
    
    X_mean = X.mean(axis=0)
    X_std = X.std(axis=0) + 1e-8
    Y_mean = Y.mean(axis=0)
    Y_std = Y.std(axis=0) + 1e-8
    
    stats = {
        'X_mean': X_mean.tolist(),
        'X_std': X_std.tolist(),
        'Y_mean': Y_mean.tolist(),
        'Y_std': Y_std.tolist()
    }
    
    X_norm = (X - X_mean) / X_std
    Y_norm = (Y - Y_mean) / Y_std
    
    split_idx = int(split_ratio * len(df))
    
    train_data = {'X_train': X_norm[:split_idx], 'Y_train': Y_norm[:split_idx]}
    val_data = {'X_val': X_norm[split_idx:], 'Y_val': Y_norm[split_idx:]}
    
    return train_data, val_data, stats

In [ ]:
# ==========================================
#         STEP 5: PINN MODEL DESIGN
# ==========================================

def build_pinn(hidden_units=128, n_layers=5):
    """
    Builds a 5-layer Multi-Layer Perceptron (MLP) with 128 hidden neurons per layer,
    using Xavier (Glorot) normal initialization and Tanh activation for differentiability.
    """
    inputs = keras.Input(shape=(2,), name='alpha_Re')
    x = inputs
    for i in range(n_layers):
        x = keras.layers.Dense(
            hidden_units,
            activation='tanh',
            kernel_initializer='glorot_normal',
            bias_initializer='zeros',
            name=f'hidden_{i+1}'
        )(x)
        
    outputs = keras.layers.Dense(
        2,
        activation='linear',
        kernel_initializer='glorot_normal',
        bias_initializer='zeros',
        name='Cd_Cl'
    )(x)
    
    return keras.Model(inputs=inputs, outputs=outputs, name='PINN_Model')

In [ ]:
# ==========================================
#   STEP 6: CUSTOM PHYSICS TRAINING LOOP
# ==========================================

def train_pinn_model(train_data, val_data, stats, epochs=3000, batch_size=64, lambda_physics=0.05):
    X_train, Y_train = train_data['X_train'], train_data['Y_train']
    X_val, Y_val = val_data['X_val'], val_data['Y_val']
    
    X_mean = tf.constant(stats['X_mean'], dtype=tf.float32)
    X_std = tf.constant(stats['X_std'], dtype=tf.float32)
    Y_mean = tf.constant(stats['Y_mean'], dtype=tf.float32)
    Y_std = tf.constant(stats['Y_std'], dtype=tf.float32)
    
    PRANDTL_LIMIT = 4.0 * math.pi
    prandtl_tf = tf.constant(PRANDTL_LIMIT, dtype=tf.float32)
    
    model = build_pinn()
    optimizer = keras.optimizers.Adam(learning_rate=1e-3)
    
    @tf.function
    def train_step(x_batch, y_batch):
        with tf.GradientTape() as tape:
            y_pred = model(x_batch, training=True)
            data_loss = tf.reduce_mean(tf.square(y_pred - y_batch))
            
            # Unnormalize outputs
            y_phys = y_pred * Y_std + Y_mean
            cd_pred = y_phys[:, 0]
            cl_pred = y_phys[:, 1]
            
            # Unnormalize alpha input
            alpha_raw = x_batch[:, 0] * X_std[0] + X_mean[0]
            
            # 1. Magnus violation: cl_pred and alpha must have matching signs
            magnus_violation = tf.nn.relu(-cl_pred * alpha_raw)
            magnus_loss = tf.reduce_mean(tf.square(magnus_violation))
            
            # 2. Zero lift at rest: alpha = 0 -> Cl = 0
            zero_lift_mask = tf.cast(tf.abs(alpha_raw) < 0.1, tf.float32)
            zero_lift_loss = tf.reduce_mean(tf.square(cl_pred * zero_lift_mask))
            
            # 3. Drag positivity: Cd >= 0
            drag_violation = tf.nn.relu(-cd_pred)
            drag_loss = tf.reduce_mean(tf.square(drag_violation))
            
            # 4. Prandtl lift ceiling: |Cl| <= 4*pi
            prandtl_violation = tf.nn.relu(tf.abs(cl_pred) - prandtl_tf)
            prandtl_loss = tf.reduce_mean(tf.square(prandtl_violation))
            
            physics_loss = magnus_loss + zero_lift_loss + drag_loss + prandtl_loss
            total_loss = data_loss + lambda_physics * physics_loss
            
        gradients = tape.gradient(total_loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        return total_loss, data_loss, physics_loss

    history = {'epoch': [], 'loss': [], 'data_loss': [], 'phys_loss': [], 'val_mse': []}
    dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    
    for epoch in range(1, epochs + 1):
        # Cosine decay scheduler
        lr = 1e-5 + (1e-3 - 1e-5) * 0.5 * (1.0 + math.cos(math.pi * (epoch - 1) / epochs))
        optimizer.learning_rate.assign(lr)
        
        shuffled = dataset.shuffle(buffer_size=1024).batch(batch_size)
        
        epoch_total = 0.0
        epoch_data = 0.0
        epoch_phys = 0.0
        batches = 0
        
        for x_batch, y_batch in shuffled:
            t_loss, d_loss, p_loss = train_step(x_batch, y_batch)
            epoch_total += t_loss.numpy()
            epoch_data += d_loss.numpy()
            epoch_phys += p_loss.numpy()
            batches += 1
            
        # Validation check
        y_val_pred = model(X_val, training=False)
        val_mse = tf.reduce_mean(tf.square(y_val_pred - Y_val)).numpy()
        
        epoch_total /= batches
        epoch_data /= batches
        epoch_phys /= batches
        
        history['epoch'].append(epoch)
        history['loss'].append(epoch_total)
        history['data_loss'].append(epoch_data)
        history['phys_loss'].append(epoch_phys)
        history['val_mse'].append(val_mse)
        
        if epoch % 200 == 0 or epoch == 1 or epoch == epochs:
            print(f"Epoch {epoch:4d}/{epochs} - loss: {epoch_total:.5f} | data: {epoch_data:.5f} | phys: {epoch_phys:.6f} | val_mse: {val_mse:.5f} | lr: {lr:.2e}")
            
    return model, history

In [ ]:
# ==========================================
#         STEP 7: RUN DATA PREPARATION & TRAINING
# ==========================================

train_data, val_data, stats = prepare_dataset(raw_literature_data, n_interp=60, noise_copies=5)
pinn_model, history = train_pinn_model(train_data, val_data, stats, epochs=3000, lambda_physics=0.05)

In [ ]:
# ==========================================
#         STEP 8: PLOT TRAINING HISTORY
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['epoch'], history['loss'], color='#185FA5')
axes[0].set_yscale('log')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total training loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['epoch'], history['data_loss'], color='#185FA5', label='Train data')
axes[1].plot(history['epoch'], history['val_mse'], color='#D95F02', linestyle='--', label='Val MSE')
axes[1].set_yscale('log')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')
axes[1].set_title('Train vs validation MSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['epoch'], history['phys_loss'], color='#1D9E75')
axes[2].set_yscale('log')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Physics loss')
axes[2].set_title('Physics constraint loss')
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
#         STEP 9: DEFINE PREDICT HELPER
# ==========================================

def predict_pinn(alphas, re_value, model, stats):
    X_mean = np.array(stats['X_mean'])
    X_std = np.array(stats['X_std'])
    Y_mean = np.array(stats['Y_mean'])
    Y_std = np.array(stats['Y_std'])
    
    X_raw = np.zeros((len(alphas), 2), dtype=np.float32)
    X_raw[:, 0] = alphas
    X_raw[:, 1] = re_value
    
    X_norm = (X_raw - X_mean) / X_std
    Y_norm = model(X_norm, training=False).numpy()
    Y_phys = Y_norm * Y_std + Y_mean
    return Y_phys[:, 0], Y_phys[:, 1]

In [ ]:
# ==========================================
#     STEP 10: PLOT Re = 140k VALIDATION
# ==========================================

dense_alpha = np.linspace(-2.2, 2.2, 100)
cd_dense, cl_dense = predict_pinn(dense_alpha, 140000, pinn_model, stats)

lit_140 = raw_literature_data['140000']
lit_140_alpha = np.array(lit_140['alpha'])
lit_140_cd = np.array(lit_140['Cd'])
lit_140_cl = np.array(lit_140['Cl'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(dense_alpha, cd_dense, label='PINN Prediction', color='#185FA5', linewidth=2)
axes[0].scatter(lit_140_alpha, lit_140_cd, color='black', marker='D', s=50, label='LES (Karabelas 2010)', zorder=5)
axes[0].set_xlabel(r'Spin ratio $\alpha$', fontsize=11)
axes[0].set_ylabel('$Cd$', fontsize=11)
axes[0].set_title(r'Drag Coefficient $Cd$ vs $\alpha$', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(dense_alpha, cl_dense, label='PINN Prediction', color='#E24B4A', linewidth=2)
axes[1].scatter(lit_140_alpha, lit_140_cl, color='black', marker='D', s=50, label='LES (Karabelas 2010)', zorder=5)
axes[1].set_xlabel(r'Spin ratio $\alpha$', fontsize=11)
axes[1].set_ylabel('$Cl$', fontsize=11)
axes[1].set_title(r'Lift Coefficient $Cl$ vs $\alpha$', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
#   STEP 10.5: OPERATING POINT POLAR & FORCE SWEEPS
# ==========================================

test_re = 1000000.0
alphas_sweep = np.linspace(-8.0, 8.0, 100)
cd_sweep, cl_sweep = predict_pinn(alphas_sweep, test_re, pinn_model, stats)
ld_sweep = cl_sweep / np.clip(cd_sweep, 0.001, None)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Polar Plot (Cl vs Cd)
axes[0].plot(cd_sweep, cl_sweep, color='#1E3A8A', linewidth=2.5)
axes[0].set_xlabel('Drag Coefficient ($Cd$)', fontsize=11)
axes[0].set_ylabel('Lift Coefficient ($Cl$)', fontsize=11)
axes[0].set_title(f'Aerodynamic Polar Curve (Re={test_re:,})', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 2. Force Coefficients vs Alpha
axes[1].plot(alphas_sweep, cd_sweep, color='#185FA5', linewidth=2.5, label='$Cd$ (Drag)')
axes[1].plot(alphas_sweep, cl_sweep, color='#E24B4A', linewidth=2.5, label='$Cl$ (Lift)')
axes[1].set_xlabel(r'Spin ratio $\alpha$', fontsize=11)
axes[1].set_ylabel('Force Coefficient', fontsize=11)
axes[1].set_title('Forces vs Spin Ratio', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Lift-to-Drag Ratio vs Alpha
axes[2].plot(alphas_sweep, ld_sweep, color='#1D9E75', linewidth=2.5)
axes[2].set_xlabel(r'Spin ratio $\alpha$', fontsize=11)
axes[2].set_ylabel('Lift-to-Drag Ratio ($Cl/Cd$)', fontsize=11)
axes[2].set_title('L/D Ratio vs Spin Ratio', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
#     STEP 11: PLOT Re = 60k VALIDATION
# ==========================================

alphas_aoki_sweep = np.linspace(0.0, 2.0, 100)
cd_aoki_pred, cl_aoki_pred = predict_pinn(alphas_aoki_sweep, 60000, pinn_model, stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot Cd
axes[0].plot(alphas_aoki_sweep, cd_aoki_pred, color='#185FA5', linewidth=2.5, label='PINN Prediction')
aoki_alpha = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
aoki_cd = np.array([1.200, 1.100, 0.900, 0.600, 0.250])
axes[0].plot(aoki_alpha, aoki_cd, color='#E24B4A', linestyle='--', marker='o', label='Aoki & Ito (2001) RANS')
exp_alpha_cd = np.array([0.05, 0.20, 0.30, 0.41, 0.50, 0.55, 0.69, 0.80, 0.92, 0.99])
exp_cd = np.array([1.13, 1.09, 1.06, 1.02, 0.98, 0.97, 0.78, 0.67, 0.59, 0.57])
axes[0].scatter(exp_alpha_cd, exp_cd, color='black', facecolors='none', edgecolors='black', s=50, label='Experimental Re=60k')
axes[0].set_xlabel(r'Spin ratio $\alpha$', fontsize=11)
axes[0].set_ylabel(r'Drag Coefficient ($Cd$)', fontsize=11)
axes[0].set_title('Drag Coefficient Comparison (Re = 60,000)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot Cl
axes[1].plot(alphas_aoki_sweep, np.abs(cl_aoki_pred), color='#185FA5', linewidth=2.5, label='PINN Prediction')
aoki_cl = np.array([0.000, 0.600, 1.500, 2.700, 3.500])
axes[1].plot(aoki_alpha, aoki_cl, color='#E24B4A', linestyle='--', marker='o', label='Aoki & Ito (2001) RANS')
exp_alpha_cl = np.array([0.13, 0.32, 0.46, 0.57, 0.68, 1.00])
exp_cl = np.array([0.09, 0.30, 0.44, 0.52, 0.44, 1.10])
axes[1].scatter(exp_alpha_cl, exp_cl, color='black', facecolors='none', edgecolors='black', s=50, label='Experimental Re=60k')
axes[1].set_xlabel(r'Spin ratio $\alpha$', fontsize=11)
axes[1].set_ylabel(r'Lift Coefficient |$Cl$|', fontsize=11)
axes[1].set_title('Lift Coefficient Comparison (Re = 60,000)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
#     STEP 12: PLOT LIFT-TO-DRAG EFFICIENCY
# ==========================================

alphas_sweep = np.linspace(0.0, 8.0, 200)
cd_sweep, cl_sweep = predict_pinn(alphas_sweep, 1000000, pinn_model, stats)
ld_ratio = cl_sweep / np.clip(cd_sweep, 0.001, None)

best_idx = np.argmax(ld_ratio)
best_alpha = alphas_sweep[best_idx]
best_ld = ld_ratio[best_idx]

plt.figure(figsize=(7, 5))
plt.plot(alphas_sweep, ld_ratio, color='#1D9E75', linewidth=2.5, label='L/D Ratio')
plt.axvline(x=best_alpha, color='#E24B4A', linestyle='--', label=f'Optimal L/D = {best_ld:.2f} at $\alpha$ = {best_alpha:.1f}')
plt.scatter(best_alpha, best_ld, color='#E24B4A', s=60, zorder=5)
plt.xlabel(r'Spin ratio $\alpha$', fontsize=11)
plt.ylabel('Lift-to-Drag Ratio ($Cl/Cd$)', fontsize=11)
plt.title('Lift-to-Drag Efficiency Curve (Re = 1,000,000)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ==========================================
#         STEP 13: PLOT DRAG CRISIS
# ==========================================

re_values = np.linspace(60000, 5000000, 500)
cd_zero = []
for re_val in re_values:
    cd_val, _ = predict_pinn([0.0], re_val, pinn_model, stats)
    cd_zero.append(cd_val[0])
    
paper_re = np.array([60000, 140000, 500000, 1000000, 5000000])
paper_cd_zero = np.array([1.20, 1.03, 0.80, 0.50, 0.30])

plt.figure(figsize=(8, 5))
plt.plot(re_values, cd_zero, color='#185FA5', linewidth=2.5, label='PINN Prediction')
plt.scatter(paper_re, paper_cd_zero, color='black', marker='D', s=50, label='Literature Data', zorder=5)
plt.xscale('log')
plt.xlabel('Reynolds Number (Log Scale)', fontsize=11)
plt.ylabel(r'Drag Coefficient ($Cd$ at $\alpha = 0$)', fontsize=11)
plt.title(r'Drag Crisis Curve for Stationary Cylinder ($\alpha = 0$)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

In [ ]:
# ==========================================
#       STEP 14: PLOT MULTI-RE SWEEPS
# ==========================================

colors_re = {
    60000: '#E24B4A',
    140000: '#185FA5',
    500000: '#1D9E75',
    1000000: '#F5A623',
    5000000: '#9B59B6'
}

alphas_full = np.linspace(-8.0, 8.0, 200)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for re_val, color in colors_re.items():
    cd_f, cl_f = predict_pinn(alphas_full, re_val, pinn_model, stats)
    re_k = re_val / 1000
    label = f'Re={re_k:.0f}k' if re_k < 1000 else f'Re={re_k/1000:.0f}M'
    
    axes[0].plot(alphas_full, cd_f, color=color, linewidth=2, label=label)
    axes[1].plot(alphas_full, cl_f, color=color, linewidth=2, label=label)
    
    re_str = str(re_val)
    if re_str in raw_literature_data:
        lit = raw_literature_data[re_str]
        lit_alphas = np.array(lit['alpha'])
        lit_cd = np.array(lit['Cd'])
        lit_cl = np.array(lit['Cl'])
        
        axes[0].scatter(lit_alphas, lit_cd, color=color, marker='o', s=50, edgecolor='black', zorder=5)
        axes[1].scatter(lit_alphas, lit_cl, color=color, marker='o', s=50, edgecolor='black', zorder=5)
        
        axes[0].scatter(-lit_alphas[lit_alphas > 0.05], lit_cd[lit_alphas > 0.05], color=color, marker='o', s=50, facecolors='none', edgecolor=color, alpha=0.6, zorder=4)
        axes[1].scatter(-lit_alphas[lit_alphas > 0.05], -lit_cl[lit_alphas > 0.05], color=color, marker='o', s=50, facecolors='none', edgecolor=color, alpha=0.6, zorder=4)
        
axes[0].set_xlabel(r'Spin ratio $\alpha$', fontsize=11)
axes[0].set_ylabel('$Cd$', fontsize=11)
axes[0].set_title('Drag Coefficients: PINN vs Literature', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel(r'Spin ratio $\alpha$', fontsize=11)
axes[1].set_ylabel('$Cl$', fontsize=11)
axes[1].set_title('Lift Coefficients: PINN vs Literature', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
#       STEP 15: PLOT VORTEX REGIME MAP
# ==========================================

alphas_grid = np.linspace(0.0, 8.0, 100)
res_grid = np.geomspace(60000, 5000000, 100)

A, R = np.meshgrid(alphas_grid, res_grid)
A_flat = A.flatten()
R_flat = R.flatten()

cd_grid, cl_grid = predict_pinn(A_flat, R_flat, pinn_model, stats)
ld_grid = cl_grid / np.clip(cd_grid, 0.001, None)
LD = ld_grid.reshape(A.shape)

plt.figure(figsize=(10, 6))
contour = plt.contourf(A, R, LD, levels=50, cmap='viridis')
cbar = plt.colorbar(contour)
cbar.set_label('Lift-to-Drag Ratio ($Cl/Cd$)', fontsize=11)

plt.axvline(x=2.0, color='white', linestyle='--', linewidth=2, alpha=0.8)
plt.axvline(x=4.0, color='white', linestyle='--', linewidth=2, alpha=0.8)

plt.text(1.0, 500000, 'Alternate Vortex\nShedding\n(Von Karman)', color='white', fontsize=9, fontweight='bold', ha='center', bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.3'))
plt.text(3.0, 500000, 'Vortex Suppression\n(Steady Asymmetric\nWake)', color='white', fontsize=9, fontweight='bold', ha='center', bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.3'))
plt.text(6.0, 500000, 'Viscous Lift\nSaturation\n(Stable Wake)', color='white', fontsize=9, fontweight='bold', ha='center', bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.3'))

plt.yscale('log')
plt.xlabel(r'Spin ratio $\alpha$', fontsize=11)
plt.ylabel('Reynolds Number (Re)', fontsize=11)
plt.title('Aerodynamic State & Vortex Regime Map (Generated from PINN)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
#     STEP 16: PLOT RECONSTRUCTED STREAMLINES GRID
# ==========================================

res_str = [200, 500000, 1000000, 5000000]
alphas_str = [0, 2, 3, 4, 5, 6, 7, 8]

fig_grid, axes_grid = plt.subplots(8, 4, figsize=(16, 24))

x_grid = np.linspace(-2.5, 3.5, 100)
y_grid = np.linspace(-2.0, 2.0, 80)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid)

for r_idx, alpha in enumerate(alphas_str):
    for c_idx, Re in enumerate(res_str):
        ax = axes_grid[r_idx, c_idx]
        
        cd_pred, cl_pred = predict_pinn([float(alpha)], float(Re), pinn_model, stats)
        cd_val = cd_pred[0]
        cl_val = cl_pred[0]
        
        u, v, v1, v2 = get_flow_velocity(X_grid, Y_grid, float(alpha), float(Re), cl_val, cd_val)
        u_masked = np.ma.masked_invalid(u)
        v_masked = np.ma.masked_invalid(v)
        
        x_inlet = -2.4 * np.ones(15)
        y_inlet = np.linspace(-1.9, 1.9, 15)
        start_pts = np.column_stack((x_inlet, y_inlet))
        
        theta_seeds = np.linspace(0, 2*np.pi, 12)
        r_seeds = [1.08, 1.25, 1.5]
        circ_pts = []
        for r in r_seeds:
            for t in theta_seeds:
                circ_pts.append([r * np.cos(t), r * np.sin(t)])
        circ_pts = np.array(circ_pts)
        
        x_wake = np.linspace(1.1, 2.2, 4)
        y_wake = np.linspace(-0.5, 0.5, 4)
        xw, yw = np.meshgrid(x_wake, y_wake)
        wake_pts = np.column_stack((xw.flatten(), yw.flatten()))
        
        seed_points = np.vstack((start_pts, circ_pts, wake_pts))
        
        ax.streamplot(x_grid, y_grid, u_masked, v_masked, start_points=seed_points, 
                      color='#1E293B', linewidth=0.6, arrowstyle='->', arrowsize=0.6, density=0.8)
        
        circle = plt.Circle((0, 0), 1.0, facecolor='#64748B', edgecolor='#1E293B', zorder=10)
        ax.add_patch(circle)
        
        if abs(alpha) < 4.0:
            ax.scatter([v1[0], v2[0]], [v1[1], v2[1]], color='#EF4444', s=15, zorder=11)
        
        if abs(alpha) <= 2.0:
            theta1 = np.arcsin(min(1.0, alpha / 2.0))
            theta2 = np.pi - theta1
            ax.scatter([1.0 * np.cos(theta1), 1.0 * np.cos(theta2)], 
                       [1.0 * np.sin(theta1), 1.0 * np.sin(theta2)], 
                       color='#F5A623', s=15, zorder=11, marker='X')
        else:
            r_stag = 1.0 * (abs(alpha) + np.sqrt(alpha**2 - 4.0)) / 2.0
            theta_stag = np.pi / 2.0 if alpha > 0 else -np.pi / 2.0
            ax.scatter([r_stag * np.cos(theta_stag)], 
                       [r_stag * np.sin(theta_stag)], 
                       color='#F5A623', s=15, zorder=11, marker='X')
        
        if Re == 200:
            if alpha in [2.0, 3.0]:
                theta1 = np.arcsin(min(1.0, alpha / 2.0))
                theta2 = np.pi - theta1
                ax.text(1.25 * np.cos(theta1), 1.25 * np.sin(theta1) - 0.1, 'L1', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
                ax.text(1.25 * np.cos(theta2), 1.25 * np.sin(theta2) - 0.1, 'L2', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
            elif alpha >= 4.0:
                r_stag = 1.0 * (abs(alpha) + np.sqrt(alpha**2 - 4.0)) / 2.0
                ax.text(0.15, min(1.6, r_stag + 0.15), 'L', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
        else:
            if alpha in [2.0, 3.0]:
                ax.text(-1.3, -0.4, 'A', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
                ax.text(-0.1, 1.2, 'B', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
                ax.text(1.5, 0.45, 'C', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
            elif alpha >= 4.0:
                ax.text(-1.3, -0.3, 'D', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
                ax.text(-0.6, 1.1, 'B', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
                r_stag = 1.0 * (abs(alpha) + np.sqrt(alpha**2 - 4.0)) / 2.0
                ax.text(0.15, min(1.6, r_stag + 0.15), 'L', color='red', fontsize=8, fontweight='bold', ha='center', va='center', zorder=15)
        
        ax.set_aspect('equal')
        ax.set_xlim(-2.2, 3.2)
        ax.set_ylim(-1.8, 1.8)
        ax.grid(True, alpha=0.1)
        
        if r_idx == 0:
            re_label = "200" if Re == 200 else f"5e5" if Re == 500000 else f"1e6" if Re == 1000000 else f"5e6"
            ax.set_title(f"Re = {re_label}", fontsize=12, fontweight='bold')
        if c_idx == 0:
            ax.set_ylabel(r"$\alpha$ = " + str(alpha), fontsize=12, fontweight='bold', labelpad=12)
        ax.text(-2.0, -1.6, f"Cd={cd_val:.2f}\nCl={cl_val:.2f}", fontsize=8, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
#     STEP 17: PLOT DIMENSIONLESS VORTICITY GRID
# ==========================================

alphas_vort = [0, 2, 3, 4, 5, 6, 7, 8]
Re_vort = 1000000.0

fig_vort, axes_vort = plt.subplots(4, 2, figsize=(14, 20))
axes_vort = axes_vort.flatten()

x_vort = np.linspace(-2.0, 5.0, 200)
y_vort = np.linspace(-2.5, 2.5, 180)
X_vort, Y_vort = np.meshgrid(x_vort, y_vort)

R_cyl = 1.0
delta_bl = 0.08 * R_cyl

for i, alpha in enumerate(alphas_vort):
    ax = axes_vort[i]
    
    cd_pred, cl_pred = predict_pinn([float(alpha)], float(Re_vort), pinn_model, stats)
    cd_val = cd_pred[0]
    cl_val = cl_pred[0]
    
    r_grid = np.sqrt(X_vort**2 + Y_vort**2)
    theta_grid = np.arctan2(Y_vort, X_vort)
    
    omega_bl = (-4.0 * np.sin(theta_grid) + 2.0 * alpha) * np.exp(-(r_grid - R_cyl)**2 / (2 * delta_bl**2))
    
    # Coordinate transformation for downstream wake bending
    x_offset = np.clip(X_vort - R_cyl, 0, None)
    deflection_max = 0.28 * alpha * (1.0 + 0.5 / np.log10(Re_vort))
    deflection_curve = deflection_max * (1.0 - np.exp(-0.45 * x_offset))
    Y_bent_vort = Y_vort - deflection_curve
    
    y_c = 0.0
    w_wake = 0.45 * R_cyl + 0.15 * np.sqrt(np.clip(X_vort - R_cyl, 0, None))
    sigma_wake = 0.12 * R_cyl + 0.05 * np.sqrt(np.clip(X_vort - R_cyl, 0, None))
    
    y_upper = y_c + w_wake
    y_lower = y_c - w_wake
    
    decay_upper = np.exp(-0.4 * (X_vort - R_cyl))
    decay_lower = np.exp(-0.4 * (X_vort - R_cyl))
    
    amp_upper = -8.0 * (cd_val + 0.25 * abs(cl_val)) * decay_upper
    amp_lower = 8.0 * (cd_val + 0.1 * abs(cl_val)) * np.exp(-0.25 * alpha) * decay_lower
    
    omega_wake_upper = amp_upper * np.exp(-(Y_bent_vort - y_upper)**2 / (2 * sigma_wake**2))
    omega_wake_lower = amp_lower * np.exp(-(Y_bent_vort - y_lower)**2 / (2 * sigma_wake**2))
    
    omega_wake = omega_wake_upper + omega_wake_lower
    blend = 1.0 - np.exp(-np.clip(X_vort - 0.5*R_cyl, 0, None) / (0.3*R_cyl))
    
    omega_total = omega_bl + omega_wake * blend
    w_star = 2.0 * np.abs(omega_total)
    w_star[r_grid < R_cyl] = np.nan
    
    max_val = 7.5 + 3.5 * alpha
    levels = np.linspace(1.0, max_val, 15)
    
    ax.set_facecolor('#0000cc')
    contour = ax.contourf(X_vort, Y_vort, w_star, levels=levels, cmap='jet', extend='max')
    
    # Add thin black contour lines matching the paper
    ax.contour(X_vort, Y_vort, w_star, levels=levels, colors='black', linewidths=0.3, alpha=0.5, zorder=5)
    
    cbar = fig_vort.colorbar(contour, ax=ax, fraction=0.03, pad=0.04)
    cbar.ax.tick_params(labelsize=8)
    
    circle = plt.Circle((0, 0), R_cyl, facecolor='white', edgecolor='red', linewidth=2.0, zorder=10)
    ax.add_patch(circle)
    
    ax.set_aspect('equal')
    ax.set_xlim(-1.5, 4.5)
    ax.set_ylim(-2.2, 2.2)
    ax.set_title(f"a = {alpha}", fontsize=14, fontstyle='italic', fontweight='bold', color='black', pad=-25, loc='center', zorder=15)
    ax.grid(True, alpha=0.15, color='black')
    
    ax.spines['bottom'].set_color('black')
    ax.spines['top'].set_color('black') 
    ax.spines['left'].set_color('black')
    ax.spines['right'].set_color('black')
    ax.tick_params(colors='black', labelsize=9)
    
plt.tight_layout()
fig_vort.patch.set_facecolor('white')
plt.show()